In [11]:
# %load_ext autoreload
%reload_ext autoreload
%autoreload 2
import cv2, numpy as np
from src.utils.font_processor import font2img, get_font_path

def cv_show(img, title='image'):
  cv2.imshow(title, img)
  cv2.waitKey(0)
  cv2.destroyAllWindows()

In [30]:
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import numpy as np
import os, sys




get_font_path(0)

font_img = font2img(get_font_path(1), "H", 1)  
cv_show(font_img, "Font Image") 

font_name is a number: 0
font_name is a number: 1


error: OpenCV(5.0.0) :-1: error: (-5:Bad argument) in function 'dilate'
> Overload resolution failed:
>  - src is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'src'


In [1]:
int(-3.8)

-3

In [15]:
from pathlib import Path

FONTS_DIR = Path("C:/Windows/Fonts")
print(FONTS_DIR / "123")

C:\Windows\Fonts\123


In [ ]:


img1 = cv2.imread("./dataset/raw/env/0072.jpg")
img2 = cv2.imread("./dataset/raw/env/0013.jpg")
dst = cv2.add(img1, img2)                    # 饱和加法，会变亮
dst = cv2.add(img1, img2, dtype=cv2.CV_32F)  # 保留高精度
cv_show(dst, "Image 1")

In [13]:
def pyr_blend(img1, img2, mask, levels=6):
  gp1 = [img1.astype(np.float32)]
  gp2 = [img2.astype(np.float32)]
  gpM = [mask.astype(np.float32)]
  for _ in range(levels):
    gp1.append(cv2.pyrDown(gp1[-1]))
    gp2.append(cv2.pyrDown(gp2[-1]))
    gpM.append(cv2.pyrDown(gpM[-1]))

  # 拉普拉斯金字塔
  lp1 = [gp1[-1]]; lp2 = [gp2[-1]]
  for i in range(levels, 0, -1):
    size = (gp1[i-1].shape[1], gp1[i-1].shape[0])
    lp1.append(cv2.subtract(gp1[i-1], cv2.pyrUp(gp1[i], dstsize=size)))
    lp2.append(cv2.subtract(gp2[i-1], cv2.pyrUp(gp2[i], dstsize=size)))

  # 按 mask 融合各层
  blended = []
  for l1, l2, m in zip(lp1, lp2, gpM[::-1]):
    m3 = np.repeat(m[..., None], 3, axis=2) if m.ndim == 2 else m
    blended.append(l1 * m3 + l2 * (1 - m3))

  # 重构
  out = blended[0]
  for i in range(1, len(blended)):
    size = (blended[i].shape[1], blended[i].shape[0])
    out = cv2.pyrUp(out, dstsize=size) + blended[i]

  return np.clip(out, 0, 255).astype(np.uint8)


# 使用
h, w = img1.shape[:2]
mask = np.zeros((h, w), np.float32)
mask[:, :w//2] = 1.0   # 左半部分取 img1
out = pyr_blend(img1, img2, mask, levels=5)
cv_show(out, "Image 1")